In [28]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../Data/raw/airtel.db')  
 
def q(sql):                                     
    return pd.read_sql(sql, con)

In [29]:
q("""SELECT name FROM sqlite_master                  
WHERE  type = 'table'                           
ORDER  BY name;""")

,name
0,circles
1,complaints
2,customers
3,plans
4,recharges
5,usage_daily


In [30]:
q("""SELECT 'circles' AS tbl, COUNT(*) AS n FROM circles        -- 22 expected
UNION ALL SELECT 'plans',       COUNT(*) FROM plans          -- 14
UNION ALL SELECT 'customers',   COUNT(*) FROM customers     -- 22,000
UNION ALL SELECT 'usage_daily', COUNT(*) FROM usage_daily    -- 1,326,739
UNION ALL SELECT 'recharges',   COUNT(*) FROM recharges      -- 408,089
UNION ALL SELECT 'complaints',  COUNT(*) FROM complaints;    -- 26,841""")

,tbl,n
0,circles,22
1,plans,14
2,customers,22000
3,usage_daily,1326739
4,recharges,408089
5,complaints,26841


In [31]:
import sqlite3                                 
import pandas as pd                             
 
pcon = sqlite3.connect('../Data/practice/practice.db')  
pd.read_sql('SELECT * FROM loans', pcon)
 
def pq(sql):                                     
    return pd.read_sql(sql, pcon)

In [32]:
pq("""SELECT * FROM shop_customers;""")

,customer_id,name,city,joined_on
0,1,Asha,Kochi,2024-01-15
1,2,Ravi,Chennai,2024-03-02
2,3,Meera,Kochi,2024-06-20
3,4,Arjun,Pune,2024-09-10
4,5,Fatima,Chennai,2025-01-05
5,6,John,Pune,2025-02-14


In [33]:
pq("""SELECT * FROM shop_orders;""")

,order_id,customer_id,order_date,amount
0,101,1,2025-01-03,450
1,102,1,2025-01-20,300
2,103,2,2025-01-11,1200
3,104,3,2025-02-02,250
4,105,1,2025-02-15,600
5,106,4,2025-02-21,900
6,107,2,2025-03-05,150
7,108,5,2025-03-09,700
8,109,3,2025-03-18,400
9,110,2,2025-03-28,350


In [34]:
pq("""SELECT * FROM members;""")

,member_id,name,joined_on
0,1,Anil,2023-07-01
1,2,Divya,2024-02-10
2,3,Karthik,2024-05-25
3,4,Lakshmi,2024-11-30
4,5,Nisha,2025-01-12
5,6,Sameer,2025-03-01


In [35]:
pq("""SELECT * FROM loans;""")

,loan_id,member_id,book,loan_date,returned_on
0,1,1,Wings of Fire,2025-01-04,2025-01-18
1,2,2,Godan,2025-01-09,2025-01-30
2,3,1,Gitanjali,2025-01-22,2025-02-05
3,4,3,Malgudi Days,2025-02-03,NaN
4,5,2,The Guide,2025-02-11,2025-02-25
5,6,4,Train to Pakistan,2025-02-19,2025-03-10
6,7,1,Discovery of India,2025-03-02,NaN
7,8,5,Ignited Minds,2025-03-06,2025-03-20
8,9,2,Wise and Otherwise,2025-03-15,NaN
9,10,3,Swami and Friends,2025-03-27,2025-04-08


In [36]:
pq("""SELECT 'shop_customers' AS tbl, COUNT(*) AS n FROM shop_customers
UNION ALL SELECT 'shop_orders', COUNT(*) FROM shop_orders
UNION ALL SELECT 'members', COUNT(*) FROM members
UNION ALL SELECT 'loans', COUNT(*) FROM loans;""")

,tbl,n
0,shop_customers,6
1,shop_orders,10
2,members,6
3,loans,10


In [37]:
#   for each customer, how many orders do they place?
# how many of those were 500(amount) or more?

pq("""SELECT c.customer_id, c.name, COUNT(o.order_id) AS order_count,
    SUM(CASE WHEN o.amount >= 500 THEN 1 ELSE 0 END) AS orders_above_500
FROM shop_customers c
LEFT JOIN shop_orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name;""")

,customer_id,name,order_count,orders_above_500
0,1,Asha,3,1
1,2,Ravi,3,1
2,3,Meera,2,0
3,4,Arjun,1,1
4,5,Fatima,1,1
5,6,John,0,0


In [38]:
pq(""" SELECT COUNT(*) AS orders,
    SUM(amount>=500) AS orders_above_500,
    ROUND(100.0*AVG(amount>=500),2) AS pct
    FROM shop_orders;""")

,orders,orders_above_500,pct
0,10,4,40.0


In [39]:
# on 31 march 2025,how many days has each customer been with the stop?
# customer name,joined data,no of days  using julianday() function

pq("""SELECT name, joined_on, CAST(julianday('2025-03-31') - julianday(joined_on) AS INTEGER) AS days_with_shop
FROM shop_customers;""")

,name,joined_on,days_with_shop
0,Asha,2024-01-15,441
1,Ravi,2024-03-02,394
2,Meera,2024-06-20,284
3,Arjun,2024-09-10,202
4,Fatima,2025-01-05,85
5,John,2025-02-14,45


In [40]:
# how many orders and how much sales in each month ?
# month, no of orders, total sales

pq(""" SELECT strftime('%Y-%m', order_date) AS month, COUNT(*) AS orders, SUM(amount) AS total_sales
    FROM shop_orders
    GROUP BY month;""")

,month,orders,total_sales
0,2025-01,3,1950
1,2025-02,3,1750
2,2025-03,4,1600


In [41]:
# which customer has never placed an order? using left join and is null

pq("""SELECT c.customer_id, c.name
FROM shop_customers c
LEFT JOIN shop_orders o ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL;""")

,customer_id,name
0,6,John


In [42]:
# which member has never taken a loan? using left join and is null

pq("""SELECT m.member_id, m.name
FROM members m
LEFT JOIN loans l ON m.member_id = l.member_id
WHERE l.member_id IS NULL;""")

,member_id,name
0,6,Sameer


In [43]:
# which customer placed at least one order on after 1st march 2025? using inner join and exists
pq("""SELECT c.customer_id, c.name
FROM shop_customers c
WHERE EXISTS (
    SELECT 1
    FROM shop_orders o
    WHERE c.customer_id = o.customer_id AND o.order_date >= '2025-03-01'
);""")


,customer_id,name
0,2,Ravi
1,3,Meera
2,5,Fatima


In [50]:
def run_sql_file(path):
    with open(path,'r') as f:
        script = f.read()
    pcon.executescript(script)
    pcon.commit()
    print('ran', path)

In [ ]:
import sqlite3
import pandas as pd

# Use the correct database
db_path = r"C:\Users\muhju\Downloads\airtel Telecom Analysis\Data\practice\practice.db"
pcon = sqlite3.connect(db_path)

def pq(sql):
    return pd.read_sql(sql, pcon)

# Check what tables really exist in this DB
print(pq("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"))

OperationalError: no such table: main.orders